In [ ]:
#1 Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

In [ ]:
#2 Loading the dataset
df = pd.read_csv("diabetes_eda.csv")

# Sanity check: confirm the impossible zeros are already NaN
print(df.isnull().sum())

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [ ]:
missing = (df.isnull().sum()/ len(df))*100
missing

Pregnancies                  0.000000
Glucose                      0.651042
BloodPressure                4.557292
SkinThickness               29.557292
Insulin                     48.697917
BMI                          1.432292
DiabetesPedigreeFunction     0.000000
Age                          0.000000
Outcome                      0.000000
dtype: float64

In [ ]:
#3 Separating Features and Target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [ ]:
#4 Train-Test Split — BEFORE any imputation
# This is the critical ordering change. Splitting first means every
# statistic used to fill missing values (medians, KNN neighbour structure)
# is learned only from the training set, and the test set never
# influences how its own — or the training set's — missing values are filled.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (614, 8)
Test shape : (154, 8)


In [ ]:
#5 Median Imputation (Glucose, BloodPressure, BMI)
# SkinThickness has been moved out of this group and into cell #6 alongside
# Insulin, since its missingness (29.56%) is far closer to Insulin's (48.70%)
# than to the genuinely low-missingness variables remaining here
# (Glucose: 0.65%, BloodPressure: 4.56%, BMI: 1.43%).
# Fit: compute the median from X_train ONLY.
median_cols = ["Glucose", "BloodPressure", "BMI"]
median_values = X_train[median_cols].median()

print("Median values (from training set only):")
print(median_values)

# Apply the SAME train-derived medians to both sets.
X_train[median_cols] = X_train[median_cols].fillna(median_values)
X_test[median_cols] = X_test[median_cols].fillna(median_values)

Median values (from training set only):
Glucose          117.0
BloodPressure     72.0
BMI               32.4
dtype: float64


In [ ]:
#6 KNN Imputation (Insulin and SkinThickness)
# Both variables exceed ~29% missingness, well past the point where median
# imputation is defensible, collapsing that much of a feature to a single
# constant would flatten a large share of its variance and weaken its
# relationship with the rest of the dataset. KNN imputation instead estimates
# each missing value from the most similar patients across the other features,
# and imputes both columns jointly in one pass, so a patient missing both
# Insulin and SkinThickness has both values filled coherently from the same
# set of nearest neighbours rather than independently.
knn_impute_cols = ["Insulin", "SkinThickness"]

# Step 1: Scale before KNN — required because KNN distance calculations
# are scale-sensitive, and Insulin's raw values (hundreds) would otherwise
# dominate the distance metric over smaller-scale features. All features are
# scaled, not just the two being imputed, since KNN's neighbour search uses
# every feature's values jointly to find the closest patients.
scaler_impute = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler_impute.fit_transform(X_train),   # fit on train only
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler_impute.transform(X_test),        # transform only — reuses train's mean/std
    columns=X_test.columns,
    index=X_test.index
)

# Step 2: KNN imputation, fit on train only
knn_imputer = KNNImputer(n_neighbors=5, weights="distance")

X_train_imputed_scaled = pd.DataFrame(
    knn_imputer.fit_transform(X_train_scaled),   # fit on train only
    columns=X_train.columns,
    index=X_train.index
)
X_test_imputed_scaled = pd.DataFrame(
    knn_imputer.transform(X_test_scaled),        # transform only — reuses train's neighbour structure
    columns=X_test.columns,
    index=X_test.index
)

# Step 3: Convert back to original scale
X_train_imputed = pd.DataFrame(
    scaler_impute.inverse_transform(X_train_imputed_scaled),
    columns=X_train.columns,
    index=X_train.index
)
X_test_imputed = pd.DataFrame(
    scaler_impute.inverse_transform(X_test_imputed_scaled),
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
#7 Final sanity check — confirm no missing values remain
print("Remaining nulls in X_train:")
print(X_train_imputed.isnull().sum())

print("\nRemaining nulls in X_test:")
print(X_test_imputed.isnull().sum())

Remaining nulls in X_train:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64

Remaining nulls in X_test:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [ ]:
#8 Saving the cleaned, split, leakage-free datasets
X_train_imputed.to_csv("X_train_clean.csv", index=False)
X_test_imputed.to_csv("X_test_clean.csv", index=False)
y_train.to_csv("y_train_clean.csv", index=False)
y_test.to_csv("y_test_clean.csv", index=False)

#### Why save the split pieces separately rather than one combined df_clean.csv:
saving a single cleaned file (like your old df_clean) tempts you — or a future notebook — into re-splitting it later, which would silently reintroduce the exact leakage issue you're fixing right now. Saving X_train_clean.csv / X_test_clean.csv separately makes the split a permanent, physical fact of your data pipeline rather than something that could accidentally get redone downstream.

Why reuse the train-derived median for the test set rather than computing the test set's own median: the test set represents data the model would never have access to at the point of imputation in a real deployment scenario. Using train-only statistics to fill both sets is what keeps this leakage-free — this is exactly the same principle as fitting StandardScaler on the training set only, just applied one step earlier in the pipeline.